In [1]:
import pandas as pd
import numpy as np

---

## Level 1 — Order IDs

Each order ID encodes three things: country code, year, and sequence number — e.g. `'US-2024-0042'`.

Quick reminder:
```python
df['col'].str.extract(r'([A-Z]+)-(\d{4})-(\d+)')   # returns a DataFrame with one column per group
df['col'].str.contains('2024')                       # returns a boolean Series
```

- Use `.str.extract()` to split `order_id` into three new columns: `country`, `year`, `seq`.
- Filter to 2024 orders using `.str.contains()`. How many are there?
- Which country has the most orders overall? What is the average order amount per country?
- Among shipped orders only, what is the total and average amount? Use `np.mean`.

In [15]:
orders = pd.DataFrame({
    'order_id': ['US-2024-0042','CA-2023-0089','US-2024-0103','GB-2024-0015',
                 'CA-2024-0067','US-2023-0201','GB-2023-0044','CA-2024-0098',
                 'US-2024-0055','GB-2024-0031'],
    'amount':   [1200, 890, 2300, 540, 1100, 760, 1850, 980, 1650, 720],
    'status':   ['shipped','pending','shipped','cancelled','shipped',
                 'pending','shipped','cancelled','shipped','pending'],
})

# Your code here

orders[['country','year','seq']] = orders['order_id'].str.extract(r'([A-Z]+)-(\d{4})-(\d+)')
d2024 = orders.loc[orders['year'].str.contains('2024')]
print(d2024.shape[0],'are 2024 orders')
print(orders.groupby('country')['amount'].sum().idxmax(),'has the most orders overall')
print(orders.groupby('country')['amount'].mean())
print('total is',sum(orders.loc[orders['status'] == 'shipped','amount']))
print('average is',np.mean(orders.loc[orders['status'] == 'shipped','amount']))


7 are 2024 orders
US has the most orders overall
country
CA     990.000000
GB    1036.666667
US    1477.500000
Name: amount, dtype: float64
total is 8100
average is 1620.0


---

## Level 2 — Staff records with `.pipe()`

Quick reminder — chain cleaning steps with `.pipe()`:
```python
def step_one(df): ...; return df
def step_two(df): ...; return df

clean = df.pipe(step_one).pipe(step_two)
```

The `staff` data below has: inconsistent name and department casing, salary as `'$85,000'` strings, remote as `'Yes'`/`'NO'` strings. Some rows look distinct but are duplicates once names are normalized.

Write three pipe functions — `normalize_text`, `parse_salary`, `add_flags` — chain them, then answer:

1. How many rows remain after dropping name-based duplicates?
2. Which department has the higher average salary?
3. What fraction of staff work remotely?
4. Is there a correlation between years of experience and salary? Use `np.corrcoef`.

In [27]:
staff = pd.DataFrame({
    'emp_id':  ['E01','E02','E03','E04','E05','E06','E07','E08'],
    'name':    ['Alice Wang','bob chen','CAROL LI','Dave Park',
                'alice wang','EVE NG','frank kim','DAVE PARK'],
    'dept':    ['R&D','SALES','r&d','Sales','R&D','SALES','r&d','sales'],
    'salary':  ['$85,000','$62,000','$91,000','$70,000',
                '$85,000','$58,000','$77,000','$70,000'],
    'yrs_exp': [5, 3, 8, 4, 5, 2, 6, 4],
    'remote':  ['Yes','No','YES','yes','Yes','NO','Yes','yes'],
})

# Your code here

def normalized_text(df):
    df['name'] = df['name'].str.lower()
    df['dept'] = df['dept'].str.lower()
    return df 

def parse_salary(df):
    df['salary'] = df['salary'].str.replace('$','', regex = False)
    df['salary'] = pd.to_numeric(df['salary'].str.replace(',','', regex = False), errors='coerce')
    return df 
def add_flag(df):
    df['remote'] = df['remote'].str.lower().map({'yes':True,'no':False})
    return df 

c = staff.copy().pipe(normalized_text).pipe(parse_salary).pipe(add_flag)
cd = c.drop_duplicates(subset=['name'])

print(cd.shape[0],'remained after dropping name based duplicates')

g = cd.groupby('dept')['salary'].mean()
print(g.idxmax(),'has higher avg salary')
print(cd['remote'].mean(),'work remotely')
cc =np.corrcoef(cd['yrs_exp'],cd['salary'])
print('correlation is ',cc[0,1])
print('highly correlated')

6 remained after dropping name based duplicates
r&d has higher avg salary
0.6666666666666666 work remotely
correlation is  0.929052087111594
highly correlated


---

## Level 3 — Monthly store revenue

Twelve months of sales data. Quick reminder:
```python
df['cumulative'] = df['col'].expanding().sum()    # running total
df['smoothed']   = df['col'].ewm(span=3).mean()   # exponentially weighted moving average
df['change']     = df['col'].diff()               # month-over-month change
```

No steps — answer these five questions:

1. Add a `cumulative_revenue` column. By which month did cumulative revenue first exceed $500,000?
2. Add a `smoothed_revenue` column using `ewm(span=3)`. Which month has the highest smoothed value?
3. Add a `mom_growth` column (month-over-month change in revenue). Which month had the biggest single-month jump?
4. Use `np.argmax` to find the index of the highest-orders month. What month was it?
5. What is the correlation between `smoothed_revenue` and `orders`? Use `np.corrcoef`.

In [45]:
monthly = pd.DataFrame({
    'month':   ['2024-01','2024-02','2024-03','2024-04','2024-05','2024-06',
                '2024-07','2024-08','2024-09','2024-10','2024-11','2024-12'],
    'revenue': [42000, 38000, 51000, 48000, 55000, 62000,
                58000, 61000, 67000, 53000, 71000, 85000],
    'orders':  [210, 195, 245, 230, 268, 295,
                280, 291, 315, 258, 340, 402],
})
monthly['month'] = pd.to_datetime(monthly['month'])

# Your code here

monthly['cumulative_revenue'] = monthly['revenue'].expanding().sum()
mm = monthly.loc[monthly['cumulative_revenue']>500000]
print('by',mm.iloc[0]['month'],'first exceeded')

monthly['smoothed_revenue'] = monthly['revenue'].ewm(span = 3).mean()
print(monthly.loc[monthly['smoothed_revenue'].idxmax(),'month'],'had the highest smoomthed avg')

monthly['mom_growth'] = monthly['revenue'].diff()
print(monthly.loc[monthly['mom_growth'].idxmax(),'month'],'had the biggest jump')
print('index is: ',np.argmax(monthly['orders']))
print(monthly.iloc[np.argmax(monthly['orders'])]['month'],'had the largest order')

cc = np.corrcoef(monthly['orders'],monthly['smoothed_revenue'])[0,1]
print(f'correlations is {cc:.2f}')
print('highly correlated')

by 2024-10-01 00:00:00 first exceeded
2024-12-01 00:00:00 had the highest smoomthed avg
2024-11-01 00:00:00 had the biggest jump
index is:  11
2024-12-01 00:00:00 had the largest order
correlations is 0.97
highly correlated
